In [ ]:
import pandas as pd
import numpy as np

# Memuat dataset
df_proyek = pd.read_csv('dataset_penjualan_kantin (2).csv')

print("DATASET BERHASIL DIMUAT")
print(df_proyek.head())

In [ ]:
# Menampilkan 5 data pertama
print("5 DATA PERTAMA:")
print(df_proyek.head())

# Informasi dataset
print("\nINFORMASI DATASET:")
df_proyek.info()

# Statistik deskriptif
print("\nSTATISTIK DESKRIPTIF:")
print(df_proyek.describe(include='all'))

# Ukuran dataset
print("\nUKURAN DATASET:")
print(df_proyek.shape)

# Mengecek missing value
print("\nMISSING VALUE:")
print(df_proyek.isnull().sum())

# Mengecek data duplikat
print("\nDATA DUPLIKAT:")
print(df_proyek.duplicated().sum())

In [ ]:
# 1. Menghapus data duplikat
df_proyek = df_proyek.drop_duplicates().copy()

# 2. Membersihkan kolom jumlah_terjual
df_proyek['jumlah_terjual'] = pd.to_numeric(
    df_proyek['jumlah_terjual']
    .astype(str)
    .str.extract(r'(\d+)')[0],
    errors='coerce'
)

# 3. Membersihkan kolom harga_satuan
df_proyek['harga_satuan'] = pd.to_numeric(
    df_proyek['harga_satuan']
    .astype(str)
    .str.replace(r'[^0-9]', '', regex=True),
    errors='coerce'
)

# 4. Membersihkan format kategori
df_proyek['kategori'] = (
    df_proyek['kategori']
    .str.strip()
    .str.title()
)

# 5. Membersihkan format tanggal
bulan = {
    'Januari': '01',
    'Februari': '02',
    'Maret': '03',
    'April': '04',
    'Mei': '05',
    'Juni': '06',
    'Juli': '07',
    'Agustus': '08',
    'September': '09',
    'Oktober': '10',
    'November': '11',
    'Desember': '12'
}

def ubah_tanggal(tanggal):
    tanggal = str(tanggal).strip()

    for nama_bulan, angka_bulan in bulan.items():
        tanggal = tanggal.replace(
            nama_bulan, angka_bulan
        )

    return pd.to_datetime(
        tanggal,
        dayfirst=True,
        format='mixed',
        errors='coerce'
    )

df_proyek['tanggal'] = df_proyek['tanggal'].apply(
    ubah_tanggal
)

# 6. Mengisi missing value jumlah terjual
median_jumlah = df_proyek['jumlah_terjual'].median()

df_proyek['jumlah_terjual'] = (
    df_proyek['jumlah_terjual']
    .fillna(median_jumlah)
)

# 7. Mengisi harga kosong berdasarkan median setiap produk
median_harga_produk = (
    df_proyek.groupby('nama_produk')['harga_satuan']
    .transform('median')
)

df_proyek['harga_satuan'] = (
    df_proyek['harga_satuan']
    .fillna(median_harga_produk)
)

# Mengisi jika masih ada harga kosong
df_proyek['harga_satuan'] = (
    df_proyek['harga_satuan']
    .fillna(df_proyek['harga_satuan'].median())
)

# 8. Mengisi nama kasir kosong
df_proyek['nama_kasir'] = (
    df_proyek['nama_kasir']
    .fillna('Tidak diketahui')
)

# 9. Memastikan tipe data numerik
df_proyek['jumlah_terjual'] = (
    df_proyek['jumlah_terjual'].astype(int)
)

df_proyek['harga_satuan'] = (
    df_proyek['harga_satuan'].astype(int)
)

# 10. Mengecek hasil cleaning
print("DATA SETELAH CLEANING:")
print(df_proyek.head())

print("\nMISSING VALUE SETELAH CLEANING:")
print(df_proyek.isnull().sum())

print("\nTIPE DATA SETELAH CLEANING:")
print(df_proyek.dtypes)

print("\nUKURAN DATA SETELAH CLEANING:")
print(df_proyek.shape)

Catatan alasan pembersihan:
- Duplikat dihapus agar transaksi yang sama tidak dihitung lebih dari satu kali.
- Format kategori diseragamkan menggunakan str.title().
- Format tanggal diseragamkan menjadi tipe data datetime.
- Data jumlah terjual yang kosong diisi menggunakan median jumlah penjualan.
- Harga kosong diisi menggunakan median harga berdasarkan nama produk.
- Nama kasir yang kosong diisi dengan Tidak diketahui.
- Tipe data jumlah dan harga diubah menjadi integer.

In [ ]:
# 1. Filtering
# Menampilkan transaksi dengan jumlah penjualan lebih dari 10
filter_data = df_proyek[
    df_proyek['jumlah_terjual'] > 10
]

print("HASIL FILTERING:")
print(filter_data)

# 2. Sorting
# Mengurutkan berdasarkan jumlah penjualan tertinggi
sort_data = df_proyek.sort_values(
    by='jumlah_terjual',
    ascending=False
)

print("\nHASIL SORTING:")
print(sort_data)

# 3. Membuat kolom turunan
df_proyek['total_pendapatan'] = (
    df_proyek['jumlah_terjual'] *
    df_proyek['harga_satuan']
)

print("\nKOLOM TOTAL PENDAPATAN:")
print(df_proyek.head())

# 4. Groupby berdasarkan produk
group_produk = df_proyek.groupby(
    'nama_produk'
).agg(
    total_terjual=('jumlah_terjual', 'sum'),
    total_pendapatan=('total_pendapatan', 'sum'),
    jumlah_transaksi=('id_transaksi', 'count')
).sort_values(
    by='total_pendapatan',
    ascending=False
)

print("\nHASIL GROUPBY PRODUK:")
print(group_produk)

# 5. Groupby berdasarkan kategori
group_kategori = df_proyek.groupby(
    'kategori'
)['total_pendapatan'].sum().sort_values(
    ascending=False
)

print("\nPENDAPATAN BERDASARKAN KATEGORI:")
print(group_kategori)

In [ ]:
# Total seluruh pendapatan
total_pendapatan = df_proyek['total_pendapatan'].sum()

# Produk dengan pendapatan tertinggi
produk_tertinggi = group_produk.index[0]

pendapatan_tertinggi = group_produk.iloc[0][
    'total_pendapatan'
]

# Kategori dengan pendapatan tertinggi
kategori_tertinggi = group_kategori.index[0]

pendapatan_kategori = group_kategori.iloc[0]

# Total seluruh produk terjual
total_terjual = df_proyek['jumlah_terjual'].sum()

print("DATA PROFILING SUMMARY")
print("======================")

print("Total transaksi:", len(df_proyek))

print("Total produk terjual:", total_terjual)

print("Total pendapatan: Rp", total_pendapatan)

print(
    "Produk dengan pendapatan tertinggi:",
    produk_tertinggi
)

print(
    "Pendapatan produk tertinggi: Rp",
    pendapatan_tertinggi
)

print(
    "Kategori dengan pendapatan tertinggi:",
    kategori_tertinggi
)

print(
    "Pendapatan kategori tertinggi: Rp",
    pendapatan_kategori
)

In [ ]:
# Menyimpan dataset yang sudah dibersihkan
df_proyek.to_csv(
    'dataset_bersih.csv',
    index=False
)

print("Dataset bersih berhasil disimpan!")ha


**Kelompok**
1. Adrienne Aurelia Sasmitha (4)
2. Lia Mitha Ridistya (16)

**Judul: Exploratory Data Analysis (EDA) – Data Penjualan Kantin Sekolah**

### 1. Pertanyaan Analisis
1. Produk apa yang menghasilkan pendapatan tertinggi?
2. Bagaimana jumlah penjualan setiap produk?
3. Kategori produk apa yang memberikan pendapatan terbesar?

### 2. Hasil Analisis
Berdasarkan pengolahan dataset, ditemukan 69 baris data dengan 8 kolom. Pada tahap inspection, terdapat 4 baris duplikat serta beberapa missing value. Setelah dilakukan cleaning, tersisa 65 baris data. Selanjutnya, dilakukan filtering, sorting, pembuatan kolom total pendapatan, dan pengelompokan data berdasarkan produk serta kategori untuk mengetahui hasil penjualannya.

### 3. Data Profiling Summary
Hasil analisis menunjukkan bahwa Roti Bakar memiliki total pendapatan tertinggi sebesar Rp3.661.000, sedangkan kategori Makanan menghasilkan pendapatan terbesar sebesar Rp5.739.000. Ditemukan pula transaksi dengan jumlah penjualan yang cukup jauh berbeda dari transaksi lainnya sehingga perlu diperiksa lebih lanjut.

### 4. Kesimpulan
Melalui proyek EDA, kami dapat memahami proses pengolahan data mulai dari pemeriksaan, pembersihan, hingga manipulasi data. Hasil pengolahan memberikan informasi mengenai produk dan kategori dengan pendapatan tertinggi. Informasi tersebut dapat digunakan sebagai bahan pertimbangan dalam pengelolaan persediaan dan penjualan kantin.